In [ ]:
from typing import TypeVar

# Minimizing a Finite State Machine with a Table

In this notebook we implement the algorithm from the section *Minimization of Finite State Machines* of the
lecture notes.  This algorithm minimizes a <span style="font-variant:small-caps;">Dfa</span> by identifying states that are *equivalent*.  The notebook
`07-Minimize.ipynb` implements the same algorithm, but it only returns the minimized <span style="font-variant:small-caps;">Dfa</span>.  In contrast, this
notebook also computes the tables that are used in the lecture notes in order to find the pairs of *separable*
states, and displays them.

## Some Type Definitions

* `Char` is an alias for `str`.
* `State` is the abstract type of the states that make up the <span style="font-variant:small-caps;">Dfa</span>.

In [ ]:
Char  = str
State = TypeVar('State')

`TransRel` is the type that describes the transition function $\delta$.

In [ ]:
TransRel = dict[tuple[State, Char], State]

`DFA` is the type that describes a <span style="font-variant:small-caps;">Dfa</span>.

In [ ]:
DFA = tuple[set[State], set[Char], TransRel, State, set[State]]

`Pair` is a pair of states.

In [ ]:
Pair = tuple[State, State]

A `SeparationTable` stores the pairs of states that are known to be *separable*.  It corresponds to the
tables shown in the lecture notes:  If the pair $(p, q)$ has been recognized as separable in step $k$ of the
algorithm, then `Table[(p, q)]` is $k$.  Pairs that are not known to be separable do not occur in the table.
As separability is symmetric, the pair $(q, p)$ is in the table whenever the pair $(p, q)$ is.

In [ ]:
SeparationTable = dict[Pair, int]

The states of the minimized <span style="font-variant:small-caps;">Dfa</span> will be  sets of states of the original <span style="font-variant:small-caps;">Dfa</span>.  These states will contain those states that are *equivalent*.

In [ ]:
SetState = frozenset[State]

In [ ]:
TransRelSet = dict[tuple[SetState, Char], SetState]

In [ ]:
Min_DFA = tuple[set[SetState], set[Char], TransRelSet, SetState, set[SetState]]

## Computing the Table

The function `cart_prod(A, B)` computes the *Cartesian product* $A \times B$ of the sets $A$ and $B$ where $A \times B$ is defined as follows:
$$ A \times B := \{ (x, y) \mid x \in A \wedge y \in B \}. $$
In order to denote the types of the elements of the sets, we define the abstract types `S` and `T`.

**Function `cart_prod(A, B)`**
- *Input:* `A` and `B` are sets.
- *Output:* The Cartesian product $A \times B$.

In [ ]:
def cart_prod[S, T](A: set[S], B: set[T]) -> set[tuple[S, T]]:
    return { (x, y) for x in A for y in B }

The function `reachable(q0, Σ, 𝛿)` takes three arguments:
* `q0` is the start state of a <span style="font-variant:small-caps;">Dfa</span>,
* `Σ`  is the alphabet.
* `𝛿`  is the transition function.  The transition function is assumed to be *complete*. `𝛿` is represented as a dictionary.

It returns the set of all states that can be reached from the start state `q0` by reading strings of characters from `Σ`.
The lecture notes assume that all states of the <span style="font-variant:small-caps;">Dfa</span> are reachable.  Therefore, the states that are not reachable
are removed before the table is computed.

The result is computed via a *fixed point* computation.

**Function `reachable(q0, Σ, 𝛿)`**
- *Input:* `q0` is the start state of a complete <span style="font-variant:small-caps;">Dfa</span>, `Σ` is its alphabet, and `𝛿` is its transition function.
- *Output:* The set of all states that can be reached from `q0`.

In [ ]:
def reachable(q0: State, Σ: set[Char], 𝛿: TransRel) -> set[State]:
    Result = { q0 }
    while True:
        NewStates = { 𝛿[p, c] for p in Result for c in Σ }
        if NewStates <= Result:
            return Result
        Result |= NewStates

The function `separation_table(Q, A, Σ, 𝛿)` takes four arguments:
* `Q` is the set of states of the <span style="font-variant:small-caps;">Dfa</span>.
* `A` is the set of all accepting states,
* `Σ`  is the alphabet.
* `𝛿` is the transition function.

  `𝛿` is represented as a dictionary.

The function computes the table of all pairs of separable states that is described in the lecture notes.
To this end, it computes the sequence of sets $V_0 \subseteq V_1 \subseteq \cdots \subseteq V_K$ of pairs of
separable states that is defined by induction:
1. Initially, all accepting states are separated from the non-accepting states:
   $$ V_0 := \bigl(A \times (Q \backslash A)\bigr) \cup \bigl((Q \backslash A) \times A\bigr). $$
2. If $q_1$ and $q_2$ are two states and there is a character $c \in \Sigma$ such that the states
   $\delta(q_1, c)$ and $\delta(q_2, c)$ are already known to be separable, then $q_1$ and $q_2$ are separable, too:
   $$ V_{k+1} := V_k \cup \bigl\{ (q_1, q_2) \bigm| \exists c \in \Sigma: \bigl(\delta(q_1, c), \delta(q_2, c)\bigr) \in V_k \bigr\}. $$

The computation stops as soon as no new pairs are found, i.e. as soon as $V_{K+1} = V_K$.
As in the lecture notes, the steps are numbered starting with $1$:  The pairs of $V_0$ are entered into the table
with the number $1$ and, for $k \geq 1$, the pairs of $V_k \backslash V_{k-1}$ are entered with the number $k+1$.
Before the step $k + 1$ is performed, the table contains exactly the pairs of $V_{k-1}$.

**Function `separation_table(Q, A, Σ, 𝛿)`**
- *Input:* `Q` is the set of states of a complete <span style="font-variant:small-caps;">Dfa</span>, `A` is its set of accepting states, `Σ` is its alphabet, and `𝛿` is its transition function.
- *Output:* A table that maps every pair of separable states to the number of the step in which this pair has been recognized as separable.

In [ ]:
def separation_table(Q: set[State], A: set[State], Σ: set[Char], 𝛿: TransRel) -> SeparationTable:
    Table = { pair: 1 for pair in cart_prod(A, Q - A) | cart_prod(Q - A, A) }
    k = 1
    while True:
        NewPairs = { (q1, q2) for q1 in Q
                              for q2 in Q
                              for c  in Σ
                              if (q1, q2) not in Table and (𝛿[q1, c], 𝛿[q2, c]) in Table
                   }
        if len(NewPairs) == 0:
            return Table
        k += 1
        for pair in NewPairs:
            Table[pair] = k

## Displaying the Tables

In [ ]:
from IPython.display import HTML

The function `state_name(q)` returns the name of the state `q` that is used when the tables and the <span style="font-variant:small-caps;">Dfa</span>s are displayed.
The states of a minimized <span style="font-variant:small-caps;">Dfa</span> are frozensets of states.  These are displayed as sets of the form
$\{q_1, \cdots, q_n\}$, where the states $q_i$ are sorted by their names.

**Function `state_name(q)`**
- *Input:* `q` is a state or a frozenset of states.
- *Output:* The name of `q` as a string.

In [ ]:
def state_name(q: object) -> str:
    if isinstance(q, frozenset):
        return '{' + ', '.join(sorted(str(p) for p in q)) + '}'
    return str(q)

The function `show_table(States, Table, k, final)` displays the table after step `k` as an HTML table.  Its rows
and columns are labelled with the states.  If the pair $(p, q)$ has been recognized as separable in one of the
steps $1$, $\cdots$, $k$, the entry in the row $p$ and the column $q$ is the number of this step.  As every state
is equivalent to itself, the entries on the diagonal are marked with the symbol $\sim$ from the beginning.  All
other entries are empty.  The list `States` determines the order of the rows and columns.

If the flag `final` is `True`, the table is the *final table* that is computed after the last step.  In this
table, every pair $(p, q)$ of states that could not be separated is marked with the symbol $\sim$, as the
states $p$ and $q$ are equivalent.

**Function `show_table(States, Table, k, final)`**
- *Input:* `States` is a list of the states of a complete <span style="font-variant:small-caps;">Dfa</span>, `Table` is a separation table for this <span style="font-variant:small-caps;">Dfa</span>, `k` is a step number, and `final` is a flag that is `False` by default.
- *Output:* An HTML object that shows the table after step `k`.  The diagonal is marked with $\sim$.  If `final` is `True`, all pairs of states that have not been separated are marked with $\sim$.

In [ ]:
def show_table(States: list[State], Table: SeparationTable, k: int, final: bool = False) -> HTML:
    style = 'border: 1px solid; padding: 2px 12px; text-align: center'
    th    = f'<th style="{style}">'
    td    = f'<td style="{style}">'
    equiv = '<span style="color: rgb(0, 100, 220); font-size: 130%; font-weight: bold">∼</span>'
    html  = '<table style="border-collapse: collapse">\n<tr>' + th + '</th>'
    for q in States:
        html += th + state_name(q) + '</th>'
    html += '</tr>\n'
    for p in States:
        html += '<tr>' + th + state_name(p) + '</th>'
        for q in States:
            if (p, q) in Table and Table[p, q] <= k:
                html += td + str(Table[p, q]) + '</td>'
            elif p == q or final:
                html += td + equiv + '</td>'
            else:
                html += td + '</td>'
        html += '</tr>\n'
    html += '</table>'
    return HTML(html)

The function `explain_step(States, Table, k, Σ, 𝛿)` explains why the pairs that are entered into the table in
step `k` are separable:
- In step $1$, every accepting state is separated from every non-accepting state.
- If $k > 1$ and the pair $(p, q)$ is entered in step $k$, then there is a character $c$ such that the pair
  $\bigl(\delta(p, c), \delta(q, c)\bigr)$ has been entered in an earlier step.

As the table is symmetric, only those pairs $(p, q)$ are explained where $p$ precedes $q$ in the list `States`.
If there are several characters that separate $p$ and $q$, the smallest of them is shown.

**Function `explain_step(States, Table, k, Σ, 𝛿)`**
- *Input:* `States` is a list of the states of a complete <span style="font-variant:small-caps;">Dfa</span>, `Table` is a separation table for this <span style="font-variant:small-caps;">Dfa</span>, `k` is a step number, `Σ` is the alphabet, and `𝛿` is the transition function.
- *Output:* A string that explains why the pairs that are entered in step `k` are separable.

In [ ]:
def explain_step(States: list[State], Table: SeparationTable, k: int, Σ: set[Char], 𝛿: TransRel) -> str:
    if k == 1:
        return 'Step 1: Every accepting state is separable from every non-accepting state.'
    result = f'Step {k}:'
    for i, p in enumerate(States):
        for q in States[i+1:]:
            if Table.get((p, q)) == k:
                c = min(c for c in Σ if Table.get((𝛿[p, c], 𝛿[q, c]), k) < k)
                p1, q1 = state_name(𝛿[p, c]), state_name(𝛿[q, c])
                result += f'\n  {state_name(p)} ≁ {state_name(q)}, because '
                result += f'𝛿({state_name(p)}, {c}) = {p1}, 𝛿({state_name(q)}, {c}) = {q1}, and {p1} ≁ {q1}.'
    return result

Two states $p$ and $q$ are *equivalent*, written $p \sim q$, iff they are not separable, i.e. iff the pair
$(p, q)$ does not occur in the final table.  The function `equivalence_class(q, Q, Table)` computes the
*equivalence class* of the state $q$, which is defined as
$$ [q]_\sim := \{ p \in Q \mid p \sim q \}. $$

**Function `equivalence_class(q, Q, Table)`**
- *Input:* `q` is a state, `Q` is the set of all states, and `Table` is the final separation table.
- *Output:* The equivalence class $[q]_\sim$ of `q`.

In [ ]:
def equivalence_class(q: State, Q: set[State], Table: SeparationTable) -> SetState:
    return frozenset({ p for p in Q if (p, q) not in Table })

The function `show_tables(F)` takes a complete <span style="font-variant:small-caps;">Dfa</span> `F`, removes its unreachable states, and computes the
separation table.  Then, for every step $k$ that finds new pairs of separable states, it explains this step
and displays the table after this step.  Finally, it displays the final table, where the pairs of states that
could not be separated are marked as equivalent, and prints the equivalence classes.

**Function `show_tables(F)`**
- *Input:* `F` is a complete <span style="font-variant:small-caps;">Dfa</span>.
- *Output:* None.  The tables are displayed as a side effect.

In [ ]:
from IPython.display import display

In [ ]:
def show_tables(F: DFA) -> None:
    Q, Σ, 𝛿, q0, A = F
    Q      = reachable(q0, Σ, 𝛿)
    States = sorted(Q, key=state_name)
    Table  = separation_table(Q, A & Q, Σ, 𝛿)
    K      = max(Table.values(), default=1)
    for k in range(1, K + 1):
        print(explain_step(States, Table, k, Σ, 𝛿))
        display(show_table(States, Table, k))
    print(f'Step {K + 1}: No new pairs of separable states are found.')
    print('Hence, the pairs of states that could not be separated are equivalent:')
    display(show_table(States, Table, K, True))
    print('The equivalence classes are:')
    for C in sorted({ equivalence_class(q, Q, Table) for q in Q }, key=state_name):
        print(f'  {state_name(C)}')

## Computing the Minimized <span style="font-variant:small-caps;">Dfa</span>

The function `minimize(F)` takes a <span style="font-variant:small-caps;">Dfa</span> `F` as its input.
Here `F` is a 5-tuple of the form
$$ F = (Q, \Sigma, \delta, q_0, A). $$
The <span style="font-variant:small-caps;">Dfa</span> $F$ has to be *complete*, i.e. $\delta(q, c)$ has to be defined for all states $q$ and all
characters $c$.  After removing the unreachable states, the function computes the separation table.
Then it returns the <span style="font-variant:small-caps;">Dfa</span>
$$ \texttt{Min}(F) := \bigl( \mathcal{Q}_\sim, \Sigma, \Delta, [q_0]_\sim, \widehat{A} \bigr), $$
that is defined in the lecture notes:
- $\mathcal{Q}_\sim := \bigl\{ [q]_\sim \bigm| q \in Q \bigr\}$ is the set of all equivalence classes,
- $\Delta\bigl([q]_\sim, c\bigr) := \bigl[\delta(q, c)\bigr]_\sim$ for all $q \in Q$ and $c \in \Sigma$, and
- $\widehat{A} := \bigl\{ [q]_\sim \bigm| q \in A \bigr\}$ is the set of accepting equivalence classes.

The function $\Delta$ is well defined:  If $p \sim q$, then we have $\delta(p, c) \sim \delta(q, c)$ for all
characters $c$, since otherwise $p$ and $q$ would be separable.

**Function `minimize(F)`**
- *Input:* `F` is a complete <span style="font-variant:small-caps;">Dfa</span>.
- *Output:* A <span style="font-variant:small-caps;">Dfa</span> with the minimal number of states that accepts the same language as `F`.  Its states are sets of equivalent states of `F`.

In [ ]:
def minimize(F: DFA) -> Min_DFA:
    Q, Σ, 𝛿, q0, A = F
    Q            = reachable(q0, Σ, 𝛿)
    Table        = separation_table(Q, A & Q, Σ, 𝛿)
    EquivClasses = { equivalence_class(q, Q, Table) for q in Q }
    newDelta: TransRelSet = {}
    for q in Q:
        for c in Σ:
            newDelta[equivalence_class(q, Q, Table), c] = equivalence_class(𝛿[q, c], Q, Table)
    newQ0        = equivalence_class(q0, Q, Table)
    newAccept    = { equivalence_class(q, Q, Table) for q in A & Q }
    return EquivClasses, Σ, newDelta, newQ0, newAccept

## Drawing a <span style="font-variant:small-caps;">Dfa</span>

In [ ]:
import graphviz as gv

The function `draw_dfa(F)` draws the <span style="font-variant:small-caps;">Dfa</span> `F` as a graph.  In contrast to the function `dfa2dot` from the
notebook `FSM-2-Dot.ipynb`, the states are labelled with their names, so that they can be compared with the
tables.  If there are several characters that lead from a state $p$ to a state $q$, the graph contains only one
edge from $p$ to $q$ that is labelled with all of these characters.

**Function `draw_dfa(F)`**
- *Input:* `F` is a <span style="font-variant:small-caps;">Dfa</span>.
- *Output:* A `graphviz` graph of `F`.

In [ ]:
def draw_dfa(F: DFA) -> gv.Digraph:
    Q, Σ, 𝛿, q0, A = F
    dot = gv.Digraph()
    dot.graph_attr['rankdir'] = 'LR'
    names = { q: f'S{n}' for n, q in enumerate(sorted(Q, key=state_name)) }
    dot.node('start', label='', width='0.1', height='0.1', style='filled', color='blue')
    dot.edge('start', names[q0])
    for q in Q:
        if q in A:
            dot.node(names[q], label=state_name(q), peripheries='2')
        else:
            dot.node(names[q], label=state_name(q))
    for p in Q:
        for q in Q:
            chars = sorted(c for c in Σ if 𝛿.get((p, c)) == q)
            if len(chars) > 0:
                dot.edge(names[p], names[q], label=','.join(chars))
    return dot

## Examples

### The Example from the Lecture Notes

The <span style="font-variant:small-caps;">Dfa</span> `F1` is the <span style="font-variant:small-caps;">Dfa</span> that is used in the lecture notes in order to explain the algorithm.  The tables
computed in the steps 1 and 2 below coincide with the tables shown in the lecture notes.

In [ ]:
delta1 = { ('q0', 'a'): 'q1', ('q0', 'b'): 'q2',
           ('q1', 'a'): 'q3', ('q1', 'b'): 'q3',
           ('q2', 'a'): 'q4', ('q2', 'b'): 'q4',
           ('q3', 'a'): 'q1', ('q3', 'b'): 'q1',
           ('q4', 'a'): 'q2', ('q4', 'b'): 'q2'
         }
F1 = { 'q0', 'q1', 'q2', 'q3', 'q4' }, { 'a', 'b' }, delta1, 'q0', { 'q3', 'q4' }

In [ ]:
draw_dfa(F1)

In [ ]:
show_tables(F1)

In [ ]:
M1 = minimize(F1)
draw_dfa(M1)

### A More Complicated Example

The <span style="font-variant:small-caps;">Dfa</span> `F2` accepts all strings over the alphabet $\{\texttt{a}, \texttt{b}\}$ that end with the string
$\texttt{abb}$, i.e. we have
$$ L(F_2) = L\bigl((\texttt{a} + \texttt{b})^* \cdot \texttt{a} \cdot \texttt{b} \cdot \texttt{b}\bigr). $$
The states of `F2` are the strings of length $3$ over the alphabet $\{\texttt{a}, \texttt{b}\}$:  In the state
$xyz$ the <span style="font-variant:small-caps;">Dfa</span> remembers the last three characters $x$, $y$, and $z$ that it has read.  If it reads the character
$c$ in the state $xyz$, it switches into the state $yzc$.  The only accepting state is the state $\texttt{abb}$.
The start state is the state $\texttt{bbb}$, i.e. initially the <span style="font-variant:small-caps;">Dfa</span> acts as if it had already read the string
$\texttt{bbb}$.  This does not change the language that is accepted, as a string $w$ ends with $\texttt{abb}$ if
and only if the string $\texttt{bbb}w$ ends with $\texttt{abb}$.

In [ ]:
Q2     = { x + y + z for x in 'ab' for y in 'ab' for z in 'ab' }
delta2 = { (q, c): q[1:] + c for q in Q2 for c in 'ab' }
F2     = Q2, { 'a', 'b' }, delta2, 'bbb', { 'abb' }

In [ ]:
draw_dfa(F2)

For this <span style="font-variant:small-caps;">Dfa</span>, three steps find new pairs of separable states.

In [ ]:
show_tables(F2)

In [ ]:
M2 = minimize(F2)
draw_dfa(M2)

The minimized <span style="font-variant:small-caps;">Dfa</span> has only four states.  In order to decide whether a string ends with $\texttt{abb}$, the
<span style="font-variant:small-caps;">Dfa</span> only needs to remember the longest suffix of the input read so far that is also a prefix of
$\texttt{abb}$.  This suffix is either $\varepsilon$, $\texttt{a}$, $\texttt{ab}$, or $\texttt{abb}$:
- $\{\texttt{bbb}\}$ corresponds to the suffix $\varepsilon$,
- $\{\texttt{aaa}, \texttt{aba}, \texttt{baa}, \texttt{bba}\}$ contains the states that end with
  $\texttt{a}$ and corresponds to the suffix $\texttt{a}$,
- $\{\texttt{aab}, \texttt{bab}\}$ contains the states that end with $\texttt{ab}$ and corresponds to the
  suffix $\texttt{ab}$, and
- $\{\texttt{abb}\}$ corresponds to the suffix $\texttt{abb}$.